# Chapter 9 §9.1: Social Media Sentiment Classifier

This notebook accompanies Chapter 9 §9.1 of *Context Engineering with DSPy*: **Social Media Sentiment Classifier**.

**Required environment variables (set in your `.env`):**

- `OPENAI_API_KEY`

## Overview

```
9.1 Social Media Sentiment Classifier

Demonstrates three context engineering patterns:
1. Deconstructing complex tasks — break "classify sentiment" into specific criteria
2. Criteria evaluation — evaluate each criterion independently with an LLM judge
3. Weighted eval metric — combine scores with business-driven weights

Failure mode: Context Confusion (Ch 1.4.6)
Technique: Control Flow + Prompt Optimization (Ch 1.5)
Agentic pattern: Prompt Chaining

The key insight: a single "classify sentiment" prompt gives the model too much
latitude. Decomposing into explicit criteria and weighting them by business
priority produces measurable improvement — and changing the weights changes
optimization pressure without touching the model or prompts.

Usage:
    python sentiment_classifier.py                  # full demo with optimization
    python sentiment_classifier.py --no-optimize    # baseline only
    python sentiment_classifier.py --post "Great product but terrible support"
```


In [ ]:
%pip install -r ../requirements.txt -q

## Imports and LM configuration

In [ ]:
import os
from dotenv import load_dotenv

import dspy
from dspy.evaluate import Evaluate

load_dotenv()
lm = dspy.LM("openai/gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY"))
dspy.configure(lm=lm)

## Signatures — decomposing "sentiment" into specific criteria

In [ ]:
class SentimentCriteria(dspy.Signature):
    """Analyze a social media post across multiple business-relevant criteria.

    Evaluate each criterion independently. Do not let one criterion
    influence another — a post can be negative sentiment but low urgency.
    """

    post: str = dspy.InputField(desc="Social media post text")

    sentiment: str = dspy.OutputField(
        desc="Overall sentiment: 'positive', 'negative', or 'neutral'"
    )
    urgency: str = dspy.OutputField(
        desc="Response urgency: 'high' (needs immediate action), 'medium', or 'low'"
    )
    purchase_intent: str = dspy.OutputField(
        desc="Purchase intent: 'high' (likely to buy/churn), 'medium', or 'low'"
    )
    topic: str = dspy.OutputField(
        desc="Primary topic: 'product', 'support', 'pricing', 'competitor', or 'general'"
    )
    brand_safety: str = dspy.OutputField(
        desc="Brand safety risk: 'safe', 'review', or 'escalate'"
    )

## Module — multi-stage analysis using ChainOfThought

In [ ]:
class SentimentAnalyzer(dspy.Module):
    """Decomposes sentiment analysis into criteria-based classification.

    Uses ChainOfThought so the model reasons through each criterion
    step by step, reducing Context Confusion errors.
    """

    def __init__(self):
        super().__init__()
        self.analyze = dspy.ChainOfThought(SentimentCriteria)

    def forward(self, post):
        return self.analyze(post=post)

## Dataset — ambiguous posts that break single-metric classifiers

In [ ]:
DATASET = [
    # Ambiguous: positive about competitor, negative about us
    {
        "post": "Switched to CompetitorX last week. Their onboarding was flawless — wish yours had been that smooth.",
        "sentiment": "negative", "urgency": "medium", "purchase_intent": "high",
        "topic": "competitor", "brand_safety": "safe",
    },
    # Sarcasm: sounds positive, actually negative
    {
        "post": "Love how your app crashes every time I try to export. Really adds excitement to my workflow.",
        "sentiment": "negative", "urgency": "high", "purchase_intent": "high",
        "topic": "product", "brand_safety": "review",
    },
    # Mixed: product praise + pricing complaint
    {
        "post": "The features are genuinely great but $99/month for a solo plan? That's tough to justify.",
        "sentiment": "neutral", "urgency": "medium", "purchase_intent": "high",
        "topic": "pricing", "brand_safety": "safe",
    },
    # Clearly positive, low urgency
    {
        "post": "Just hit 10k users on your platform. The analytics dashboard is chef's kiss.",
        "sentiment": "positive", "urgency": "low", "purchase_intent": "low",
        "topic": "product", "brand_safety": "safe",
    },
    # Urgent support issue
    {
        "post": "Production is down because your API is returning 500s. We need this fixed NOW.",
        "sentiment": "negative", "urgency": "high", "purchase_intent": "high",
        "topic": "support", "brand_safety": "escalate",
    },
    # Neutral question
    {
        "post": "Does the enterprise plan include SSO? Can't find it in the docs.",
        "sentiment": "neutral", "urgency": "low", "purchase_intent": "medium",
        "topic": "pricing", "brand_safety": "safe",
    },
    # Positive but with churn signal
    {
        "post": "Your tool was great for our startup phase but we've outgrown it. Any enterprise features planned?",
        "sentiment": "positive", "urgency": "medium", "purchase_intent": "high",
        "topic": "product", "brand_safety": "safe",
    },
    # Backhanded compliment
    {
        "post": "Surprised this actually worked on the first try. Not what I expected based on the reviews.",
        "sentiment": "positive", "urgency": "low", "purchase_intent": "low",
        "topic": "product", "brand_safety": "safe",
    },
    # Brand safety risk
    {
        "post": "Your product literally ruined my presentation in front of 200 people. Sharing this everywhere.",
        "sentiment": "negative", "urgency": "high", "purchase_intent": "high",
        "topic": "support", "brand_safety": "escalate",
    },
    # Competitor comparison
    {
        "post": "Ran a side-by-side test: your API is 3x faster than CompetitorY but their docs are way better.",
        "sentiment": "neutral", "urgency": "low", "purchase_intent": "medium",
        "topic": "competitor", "brand_safety": "safe",
    },
]

def build_dataset():
    """Convert raw data into DSPy Examples with train/test split."""
    examples = [
        dspy.Example(**item).with_inputs("post")
        for item in DATASET
    ]
    return examples[:7], examples[7:]

## Criteria evaluation — individual scoring functions

In [ ]:
def sentiment_score(example, pred, trace=None):
    """Binary: did we get the sentiment right?"""
    return float(pred.sentiment.strip().lower() == example.sentiment.lower())

def urgency_score(example, pred, trace=None):
    """Asymmetric: missing a high-urgency post is much worse than over-escalating.

    Missing high urgency → -0.5 (actively harmful to customer relationships)
    Over-escalating low to high → 0.3 (wasteful but not damaging)
    """
    pred_val = pred.urgency.strip().lower()
    gold_val = example.urgency.lower()

    if pred_val == gold_val:
        return 1.0
    if gold_val == "high" and pred_val == "low":
        return -0.5  # dangerous miss
    if gold_val == "low" and pred_val == "high":
        return 0.3  # over-escalation, acceptable
    return 0.0

def purchase_intent_score(example, pred, trace=None):
    """Binary match on purchase intent."""
    return float(pred.purchase_intent.strip().lower() == example.purchase_intent.lower())

def topic_score(example, pred, trace=None):
    """Binary match on topic classification."""
    return float(pred.topic.strip().lower() == example.topic.lower())

def brand_safety_score(example, pred, trace=None):
    """Asymmetric: missing an 'escalate' is worse than false alarm.

    Missing escalation → -1.0 (PR risk)
    False escalation → 0.3 (extra review, acceptable cost)
    """
    pred_val = pred.brand_safety.strip().lower()
    gold_val = example.brand_safety.lower()

    if pred_val == gold_val:
        return 1.0
    if gold_val == "escalate" and pred_val == "safe":
        return -1.0  # missed escalation
    if gold_val == "safe" and pred_val == "escalate":
        return 0.3  # false alarm
    return 0.0

## Weighted evaluation metric — business priorities as weights

In [ ]:
# These weights encode what the business actually cares about.
# Changing them changes optimization pressure without touching the model.
WEIGHTS = {
    "brand_safety": 0.30,   # PR incidents are expensive
    "urgency":      0.25,   # SLA breaches cost money
    "sentiment":    0.20,   # core classification task
    "purchase_intent": 0.15,  # revenue signal
    "topic":        0.10,   # routing accuracy
}

SCORERS = {
    "brand_safety": brand_safety_score,
    "urgency": urgency_score,
    "sentiment": sentiment_score,
    "purchase_intent": purchase_intent_score,
    "topic": topic_score,
}

def weighted_metric(example, pred, trace=None):
    """Combine criteria scores with business-driven weights.

    Returns a float in roughly [0, 1]. Negative subscores (from missed
    urgency/safety) can push the total below zero for truly bad predictions.
    """
    scores = {name: fn(example, pred, trace) for name, fn in SCORERS.items()}
    total = sum(WEIGHTS[name] * max(0, scores[name]) for name in WEIGHTS)
    return total

def detailed_scores(example, pred):
    """Print per-criterion breakdown for debugging."""
    scores = {name: fn(example, pred) for name, fn in SCORERS.items()}
    combined = weighted_metric(example, pred)

    print(f"  Sentiment:       {scores['sentiment']:.2f} (weight {WEIGHTS['sentiment']:.0%})")
    print(f"  Urgency:         {scores['urgency']:.2f} (weight {WEIGHTS['urgency']:.0%})")
    print(f"  Purchase Intent: {scores['purchase_intent']:.2f} (weight {WEIGHTS['purchase_intent']:.0%})")
    print(f"  Topic:           {scores['topic']:.2f} (weight {WEIGHTS['topic']:.0%})")
    print(f"  Brand Safety:    {scores['brand_safety']:.2f} (weight {WEIGHTS['brand_safety']:.0%})")
    print(f"  Combined:        {combined:.3f}")
    return combined

## Run the demo

The code below mirrors the `main()` block in the source script with the argparse CLI branches removed — it runs the full demo path end-to-end.

In [ ]:
trainset, testset = build_dataset()
analyzer = SentimentAnalyzer()

# --- Baseline ---
print("=" * 60)
print("BASELINE (no optimization)")
print("=" * 60)

evaluator = Evaluate(devset=testset, metric=weighted_metric, display_progress=True)
baseline_score = float(evaluator(analyzer))
print(f"\nBaseline weighted score: {baseline_score:.1f}%")

# Show per-criterion breakdown on one example
print(f"\nDetailed breakdown on test example:")
print(f"  Post: {testset[0].post[:80]}...")
pred = analyzer(post=testset[0].post)
detailed_scores(testset[0], pred)

# --- Optimization ---
print("\n" + "=" * 60)
print("OPTIMIZATION (BootstrapFewShot with weighted metric)")
print("=" * 60)

from dspy.teleprompt import BootstrapFewShot

optimizer = BootstrapFewShot(
    metric=weighted_metric,
    max_bootstrapped_demos=3,
    max_labeled_demos=3,
)

optimized = optimizer.compile(SentimentAnalyzer(), trainset=trainset)
optimized_score = float(evaluator(optimized))

print(f"\nBaseline:  {baseline_score:.1f}%")
print(f"Optimized: {optimized_score:.1f}%")
print(f"Delta:     {optimized_score - baseline_score:+.1f}%")

# Show the same example with the optimized model
print(f"\nOptimized breakdown on same test example:")
opt_pred = optimized(post=testset[0].post)
detailed_scores(testset[0], opt_pred)

# --- Demonstrate weight sensitivity ---
print("\n" + "=" * 60)
print("WEIGHT SENSITIVITY")
print("=" * 60)
print("Same model, different business priorities:\n")

alt_weights = {
    "brand_safety": 0.10,
    "urgency":      0.10,
    "sentiment":    0.50,  # marketing team cares about sentiment most
    "purchase_intent": 0.20,
    "topic":        0.10,
}

def alt_metric(example, pred, trace=None):
    scores = {name: fn(example, pred, trace) for name, fn in SCORERS.items()}
    return sum(alt_weights[name] * max(0, scores[name]) for name in alt_weights)

alt_evaluator = Evaluate(devset=testset, metric=alt_metric, display_progress=False)

baseline_alt = float(alt_evaluator(analyzer))
optimized_alt = float(alt_evaluator(optimized))

print(f"  Original weights → Baseline: {baseline_score:.1f}%, Optimized: {optimized_score:.1f}%")
print(f"  Marketing weights → Baseline: {baseline_alt:.1f}%, Optimized: {optimized_alt:.1f}%")
print("\n  Same model, same data — different weights, different scores.")
print("  The weights are the lever, not the model.")